# Chapitre 5 · Un peu de hasard (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du notebook
du chapitre. Le code de la leçon (et le bonus MiniLM), lui, vit dans le notebook
du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

In [ ]:
# Mise en place (reprise de la lecon) : les imports et les scores du fil rouge.
import random

import torch
import torch.nn.functional as F

scores = torch.tensor([2.0, 1.0, 0.1])   # un score par candidat : i, o, u

### Exercice 1 · Le softmax à la main — niveau ●

Les deux gestes qui transforment des scores bruts (logits) en distribution de
probabilités. Réécris-les dans `ton_softmax`, sans recopier `softmax_maison` :
exponentielle d'abord, division par la somme ensuite.

In [ ]:
def ton_softmax(scores):
    """Transforme des scores bruts (logits) en distribution de probabilites."""
    exp_scores = torch.exp(scores)          # geste 1 : tout devient positif
    return exp_scores / exp_scores.sum()    # geste 2 : la somme vaut 1

In [ ]:
# Validation du softmax : les valeurs calculees a la main dans le chapitre.
probas_ex = ton_softmax(scores)
assert isinstance(probas_ex, torch.Tensor), "ton_softmax doit renvoyer un tenseur"
attendu = torch.tensor([0.6590, 0.2424, 0.0986])   # e^2/11.21, e^1/11.21, e^0.1/11.21
assert torch.allclose(probas_ex, attendu, atol=1e-3), f"attendu {attendu.tolist()}, obtenu {probas_ex.tolist()}"
assert abs(probas_ex.sum().item() - 1.0) < 1e-6, "la somme des probabilites doit valoir 1"
assert (probas_ex > 0).all(), "toutes les probabilites doivent etre strictement positives"
assert torch.allclose(probas_ex, F.softmax(scores, dim=-1), atol=1e-6), "doit coincider avec F.softmax"
print("Softmax OK :", [round(p, 4) for p in probas_ex.tolist()], "| somme =", round(probas_ex.sum().item(), 6))

### Exercice 2 · La molette temperature — niveau ●

Une seule ligne : divise les scores par `T` avant d'appeler `ton_softmax`.
$T < 1$ pique la distribution, $T > 1$ l'aplatit.

In [ ]:
def ton_softmax_temperature(scores, T):
    """Le softmax avec sa molette : divise les scores par T avant le softmax."""
    return ton_softmax(scores / T)

In [ ]:
# Validation de la temperature : le tableau du chapitre, au dixieme de pourcent.
piquee  = ton_softmax_temperature(scores, T=0.5)
neutre  = ton_softmax_temperature(scores, T=1.0)
aplatie = ton_softmax_temperature(scores, T=2.0)
assert torch.allclose(piquee,  torch.tensor([0.8638, 0.1169, 0.0193]), atol=1e-3), f"T=0.5 : obtenu {piquee.tolist()}"
assert torch.allclose(neutre,  F.softmax(scores, dim=-1), atol=1e-5), "T=1 doit redonner le softmax tel quel"
assert torch.allclose(aplatie, torch.tensor([0.5017, 0.3043, 0.1940]), atol=1e-3), f"T=2 : obtenu {aplatie.tolist()}"
print("Temperature OK :")
for nom, T, p in [("piquee ", 0.5, piquee), ("neutre ", 1.0, neutre), ("aplatie", 2.0, aplatie)]:
    print(f"  T = {T} ({nom}) : " + "  ".join(f"{c} {v*100:4.1f} %" for c, v in zip("iou", p.tolist())))

### Exercice 3 · La cross-entropy à la main — niveau ●●

La formule du chapitre, mot à mot, sur un batch de logits : softmax ligne par
ligne, lecture de la probabilité de la bonne réponse dans chaque ligne, puis
moins-log et moyenne.

In [ ]:
def ta_cross_entropy(logits, cibles):
    """La formule du chapitre, mot a mot, sur un batch de logits."""
    probas = F.softmax(logits, dim=1)                # chaque ligne devient une distribution
    p_bonnes = probas[range(len(cibles)), cibles]    # la proba donnee a la bonne reponse, par ligne
    return -torch.log(p_bonnes).mean()               # -log, puis moyenne : la formule, mot a mot

In [ ]:
# Validation de la cross-entropy : le batch de trois exemples du chapitre.
logits_ex = torch.tensor([[2.0, 1.0, 0.1],     # exemple 1 : la bonne reponse est i (indice 0)
                          [0.2, 2.3, 0.5],     # exemple 2 : la bonne reponse est o (indice 1)
                          [1.0, 1.0, 1.0]])    # exemple 3 : la bonne reponse est u (indice 2)
cibles_ex = torch.tensor([0, 1, 2])            # les indices des bonnes reponses

loss_ex = ta_cross_entropy(logits_ex, cibles_ex)
assert isinstance(loss_ex, torch.Tensor), "ta_cross_entropy doit renvoyer un tenseur"
assert abs(loss_ex.item() - 0.5895) < 1e-3, f"loss attendue 0.5895, obtenue {loss_ex.item():.4f}"
assert torch.allclose(loss_ex, F.cross_entropy(logits_ex, cibles_ex), atol=1e-5), \
    "doit coincider avec F.cross_entropy (qui contient deja le softmax)"
print(f"Cross-entropy OK : maison = {loss_ex.item():.4f} | F.cross_entropy = {F.cross_entropy(logits_ex, cibles_ex).item():.4f}")

### Exercice 4 · Le tirage au sort — niveau ●●●

La roue de loterie dépliée : découpe le segment $[0, 1)$ en zones dont les
tailles sont les probabilités (les bornes sont les **sommes cumulées**), lance
une fléchette uniforme, et renvoie la zone où elle tombe.

In [ ]:
def tirer_au_sort(probas):
    """Tire un indice au sort, proportionnellement aux probabilites."""
    r = random.random()             # la flechette : uniforme entre 0 et 1
    cumul = 0.0
    for i, p in enumerate(probas):
        cumul += p                  # la borne droite de la zone du candidat i
        if r < cumul:               # la flechette est tombee dans cette zone
            return i
    return len(probas) - 1          # filet de securite (arrondis flottants)

In [ ]:
# Validation du tirage : cas certains, puis 10 000 tirages comptes.
assert tirer_au_sort([1.0, 0.0, 0.0]) == 0, "toute la masse sur i : i doit toujours sortir"
assert tirer_au_sort([0.0, 0.0, 1.0]) == 2, "toute la masse sur u : u doit toujours sortir"

random.seed(0)                                     # flechettes reproductibles
tirages_ex = [tirer_au_sort([0.659, 0.242, 0.099]) for _ in range(10_000)]
comptes_ex = [tirages_ex.count(i) for i in range(3)]
assert comptes_ex == [6597, 2460, 943], f"avec random.seed(0), comptes attendus [6597, 2460, 943], obtenus {comptes_ex}"
for c, n in zip("iou", comptes_ex):
    print(f"« {c} » : {n:5d} tirages sur 10 000  ({n / 100:.1f} %)")
print("Echantillonnage OK : le hasard obeit aux probabilites (66 % / 24 % / 10 % environ).")